<a href="https://colab.research.google.com/github/alicsrsustain-sudo/HVAC-Optimization-/blob/main/Heat_Exchanger_Valve_Leak_3_St_Paul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  PHE-BF — Annual Cost Saving Calculator
#  3 St. Paul's Place, Sheffield


# ── CELL 1 — Install / import libraries ──────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.colab import files

print("✓ Libraries ready")


# ── CELL 2 — Upload your CSV ──────────────────────────────────────────────────
print("Select your PEAK CSV export file...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"✓ Uploaded: {filename}")


# ── CELL 3 — Load and parse CSV ───────────────────────────────────────────────
df_raw = pd.read_csv(filename, header=None, skiprows=1)
df_raw.columns = ["label", "id", "timestamp", "working_hours", "value", "unit"]
df_raw["label"]     = df_raw["label"].str.strip()
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])
df_raw["value"]     = pd.to_numeric(df_raw["value"], errors="coerce")

print(f"✓ Loaded {len(df_raw):,} rows")
print(f"  Date range : {df_raw['timestamp'].min()} → {df_raw['timestamp'].max()}")
print(f"\n  Sensor streams found:")
for lbl in df_raw["label"].unique():
    n = df_raw[df_raw["label"] == lbl]["value"].count()
    print(f"    {lbl}  ({n:,} readings)")


# ── CELL 4 — ASSUMPTIONS ─────────────────────────────────────────────────────

FLOW_RATE_KG_S          = 2.6    # ← Primary HW mass flow rate (kg/s)

SPECIFIC_HEAT_KJ_KG_K   = 4.18  # Water Cp

HEAT_COST_PER_KWH        = 0.08  # ← District heat cost (£/kWh)

VALVE_CLOSED_THRESHOLD   = 5.0   # ← Valve position (%) below which valve is closed
LEAKAGE_DELTA_DEADBAND   = 2.0   # ← Min excess °C to count as active waste
OPERATING_HOURS_PER_YEAR = 3120  # 260 days × 12 hrs

print("✓ Assumptions set")
print(f"   Flow rate        : {FLOW_RATE_KG_S} kg/s        ← ESTIMATION")
print(f"   Heat cost        : £{HEAT_COST_PER_KWH}/kWh     ← ESTIMATION")


# ── CELL 5 — Extract and align sensor streams ─────────────────────────────────
def extract_stream(df, keyword, exclude=None):
    mask = df["label"].str.contains(keyword, case=False, na=False)
    if exclude:
        mask &= ~df["label"].str.contains(exclude, case=False, na=False)
    subset = df[mask][["timestamp", "value"]].dropna()
    if subset.empty:
        raise ValueError(f"No sensor found matching: '{keyword}'"
                         + (f" (excluding '{exclude}')" if exclude else "")
                         + f"\nAvailable labels:\n" +
                         "\n".join(f"  - {l}" for l in df["label"].unique()))
    return subset.set_index("timestamp")["value"].resample("15min").mean()

lwt    = extract_stream(df_raw, "Leaving Water Temperature", exclude="Setpoint")
lwt_sp = extract_stream(df_raw, "Setpoint")
valve  = extract_stream(df_raw, "Valve Position")

merged = pd.DataFrame({
    "lwt":   lwt,
    "lwt_sp": lwt_sp,
    "valve": valve,
}).dropna()

interval_hours = 0.25
period_hours   = len(merged) * interval_hours
period_days    = period_hours / 24

print(f"✓ Aligned dataset: {len(merged):,} intervals  ({period_days:.1f} days)")


# ── CELL 6 — Identify leakage and calculate waste ────────────────────────────
merged["valve_closed"]   = merged["valve"] <= VALVE_CLOSED_THRESHOLD
merged["excess_delta_t"] = np.maximum(merged["lwt"] - merged["lwt_sp"], 0)
merged["leaking"]        = (
    merged["valve_closed"] &
    (merged["excess_delta_t"] > LEAKAGE_DELTA_DEADBAND)
)

# Wasted power (kW) = ṁ × Cp × ΔT_excess  ← ESTIMATION depends on flow rate
merged["wasted_kw"]  = FLOW_RATE_KG_S * SPECIFIC_HEAT_KJ_KG_K * merged["excess_delta_t"]
merged["wasted_kwh"] = merged["wasted_kw"] * interval_hours

# Totals for the dataset period
n_total          = len(merged)
n_leak           = merged["leaking"].sum()
leak_pct         = n_leak / n_total * 100
leak_hours       = n_leak * interval_hours
avg_wasted_dt    = merged.loc[merged["leaking"], "excess_delta_t"].mean() if n_leak > 0 else 0
avg_wasted_kw    = merged.loc[merged["leaking"], "wasted_kw"].mean() if n_leak > 0 else 0
total_wasted_kwh = merged.loc[merged["leaking"], "wasted_kwh"].sum()

# Annualise
annual_factor        = OPERATING_HOURS_PER_YEAR / period_hours
annual_heat_kwh      = total_wasted_kwh * annual_factor
annual_cost_saving   = annual_heat_kwh * HEAT_COST_PER_KWH

print("=" * 60)
print("  PHE-BF — COST SAVING RESULTS")
print("  3 St. Paul's Place, Sheffield")
print("=" * 60)
print(f"\n  Dataset period           : {merged.index.min().date()} → {merged.index.max().date()}")
print(f"  Period length            : {period_days:.1f} days ({period_hours:.0f} hrs)")
print(f"\n  Leakage intervals        : {n_leak:,} of {n_total:,}  ({leak_pct:.1f}%)")
print(f"  Leakage hours in period  : {leak_hours:.1f} hrs")
print(f"  Avg wasted delta-T       : {avg_wasted_dt:.1f} °C  (LWT above setpoint)")
print(f"  Avg wasted power         : {avg_wasted_kw:.1f} kW       ← ESTIMATION (flow rate assumed)")
print(f"  Total heat wasted        : {total_wasted_kwh:,.0f} kWh in period")
print(f"\n  Annualisation factor     : ×{annual_factor:.1f}")
print(f"  Annual heat wasted       : {annual_heat_kwh:,.0f} kWh/yr")
print(f"\n{'─'*60}")
print(f"   Est. annual cost saving : £{annual_cost_saving:,.0f}/yr")
print(f"{'─'*60}")

✓ Libraries ready
Select your PEAK CSV export file...


Saving History 2025-12-16T00_00_00 to 2026-06-17T12_43_14.csv to History 2025-12-16T00_00_00 to 2026-06-17T12_43_14 (1).csv
✓ Uploaded: History 2025-12-16T00_00_00 to 2026-06-17T12_43_14 (1).csv
✓ Loaded 51,499 rows
  Date range : 2025-12-16 00:00:00 → 2026-06-17 13:43:00

  Sensor streams found:
    PHE-BF, HE Primary Leaving Water Temperature, °C, BF, BF  (17,165 readings)
    PHE-BF, HE Primary Leaving Water Temperature Setpoint, °C, BF, BF  (17,165 readings)
    PHE-BF, Heat Exchanger Primary Entering Water Valve Position, %, BF, BF  (17,165 readings)
    Alert - HEX00024, 1085562  (4 readings)
✓ Assumptions set
   Flow rate        : 2.6 kg/s        ← ESTIMATION
   Heat cost        : £0.08/kWh     ← ESTIMATION
✓ Aligned dataset: 17,165 intervals  (178.8 days)
  PHE-BF — COST SAVING RESULTS
  3 St. Paul's Place, Sheffield

  Dataset period           : 2025-12-16 → 2026-06-17
  Period length            : 178.8 days (4291 hrs)

  Leakage intervals        : 4,058 of 17,165  (23.6%)
  L